# M11.2 test — Hub 09_2B pooled (GAP) model inference

Plan: [`plans/milestone_11/11_huggingface_artifacts_plan.md`](../../plans/milestone_11/11_huggingface_artifacts_plan.md).

Downloads the published model [`tbhugging/camera_orbit_compact_09_2b`](https://huggingface.co/tbhugging/camera_orbit_compact_09_2b) and runs a forward pass on six packaged demo `anomaly_ref` views (nearest-neighbour matches to the trained 60° camera orbit from the 36°-stride `m8_demo` corpus).

**Live Hub only:** always fetches from the remote repo (no Hugging Face cache fallback). Requires network; if the Hub is unreachable the download cell fails with an explicit error after a timeout.

In [ ]:
from pathlib import Path
import shutil
import sys
import subprocess

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

for name in ("build", "dist"):
    shutil.rmtree(ROOT / name, ignore_errors=True)
for egg in (ROOT / "src").glob("*.egg-info"):
    shutil.rmtree(egg, ignore_errors=True)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        f"{ROOT}[dl,hf,dev]",
        "-c",
        str(ROOT / "requirements.txt"),
    ]
)

from gummybear.paths import display_path, install_display_safe_warning_paths

install_display_safe_warning_paths()

print(f"ROOT={display_path(ROOT)}")

## Download published weights from the Hub

In [ ]:
from IPython.display import Markdown, display

from tomography_ml_validation.milestone_11 import (
    DEFAULT_HUB_DOWNLOAD_TIMEOUT_S,
    download_camera_orbit_compact_09_2b,
    load_camera_orbit_compact_09_2b,
    load_packaged_m9_demo_multiview_example,
    run_packaged_m9_demo_inference,
)

HUB_ID = "tbhugging/camera_orbit_compact_09_2b"
snap = download_camera_orbit_compact_09_2b(
    hub_id=HUB_ID, timeout_s=DEFAULT_HUB_DOWNLOAD_TIMEOUT_S
)
loaded = load_camera_orbit_compact_09_2b(snap, hub_id=HUB_ID)
display(Markdown(
    f"Loaded [`{loaded.hub_id}`](https://huggingface.co/{loaded.hub_id}) from "
    f"`{display_path(loaded.snapshot_dir)}`  \n"
    f"protocol=`{loaded.config.get('protocol')}`  "
    f"architecture=`{loaded.config.get('architecture')}`  "
    f"n_views={loaded.config.get('n_views')}  n_params={loaded.n_params}"
))

## Inference on packaged demo anomaly (6-view orbit)

Uses `bear_m8_high_000004` from the packaged M8 demo corpus. Demo frames are on a 36° stride; the loader picks nearest neighbours to the trained orbit `[0, 60, 120, 180, 240, 300]°` (illustrative only).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

view_angles = tuple(float(a) for a in loaded.config["view_angles_deg"])
sample_mv = load_packaged_m9_demo_multiview_example(view_angles_deg=view_angles)
result = run_packaged_m9_demo_inference(loaded)

n_views = sample_mv.views_vchw.shape[0]
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, idx in zip(axes.ravel(), range(n_views)):
    img = np.asarray(sample_mv.views_vchw[idx, 0], dtype=np.float32)
    vmin, vmax = np.percentile(img, [1, 99])
    ax.imshow(img, cmap="gray", vmin=vmin, vmax=vmax)
    req = sample_mv.view_angles_deg[idx]
    got = sample_mv.matched_angles_deg[idx]
    ax.set_title(f"req {req:g}° → demo {got:g}°")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f"{sample_mv.sequence_id}  anomaly_ref (nearest demo frames)")
plt.tight_layout()
plt.show()

yp = result.y_pred
yt = result.y_true
display(Markdown(
    f"**pred xyz** = `({yp[0]:.4f}, {yp[1]:.4f}, {yp[2]:.4f})`  \n"
    f"**true xyz** = `({yt[0]:.4f}, {yt[1]:.4f}, {yt[2]:.4f})`  \n"
    f"**Euclidean error** = `{result.euclidean_error:.4f}`  \n"
    f"Manifest: `{display_path(sample_mv.manifest_path)}`"
))

## Notes

- This notebook tests the **published Hugging Face Hub repo only** (live download; no cache fallback). Offline mode, missing network, or a hung request fail with `HubDownloadError` after `DEFAULT_HUB_DOWNLOAD_TIMEOUT_S` (30 s).
- Published weights are **09_2B pooled (GAP)** only — not the 09_2A Fourier variant.
- Demo sequence is **not** from the full M8 train/val/test split; error magnitude is illustrative only.